# WarehousePG Backups and Disaster Recovery
WarehousePG Disaster Recovery (whpg-dr) protects WarehousePG (WHPG) clusters against data loss and extended outages. It takes consistent physical backups of the entire cluster, ships WAL to a secondary storage repository, so you can restore the cluster to any named restore point, on the original hosts or on different infrastructure.

## Init
* Set variable names
* Remove any leftover backups from previous demos
* Remove any existing configuration files
* Connect to WarehousePG

In [ ]:
# Variables for demo
cluster_name = "dr_demo_cluster"
bucket_name = "warehousepg-backups"
prefix = "demo-dr-backups"
region = "us-east-1"
folder = "whpg-demo"

# Cleanup any existing backups 
!aws s3 rm --recursive s3://{bucket_name}/{prefix}
!rm -rf /home/gpadmin/.whpg-dr

# Connect to the database
from sqlalchemy import create_engine
PGUSER="gpadmin"
PGHOST="cdw"
PGPORT="5432"
PGDATABASE="dev"
conn = create_engine(f"postgresql://{PGUSER}@{PGHOST}:{PGPORT}/{PGDATABASE}")

# If running elsewhere without trust auth, set a password and use this instead:
# PGPASSWORD="your_password_here"
# conn = create_engine(f"postgresql://{PGUSER}:{PGPASSWORD}@{PGHOST}:{PGPORT}/{PGDATABASE}")

%reload_ext sql
%sql conn

## Metadata
Capture metadata about the cluster to files for the demo.

In [ ]:
result = %sql SELECT DISTINCT hostname FROM gp_segment_configuration WHERE role = 'p';
with open("all_nodes.txt", "w") as f:
    for row in result:
        f.write(row.hostname + "\n")
print(open("all_nodes.txt").read())

In [ ]:
result = %sql SELECT DISTINCT hostname FROM gp_segment_configuration WHERE role = 'p' AND content >= 0;
with open("segment_nodes.txt", "w") as f:
    for row in result:
        f.write(row.hostname + "\n")

print(open("segment_nodes.txt").read())

In [ ]:
result = %sql SELECT DISTINCT'/' || split_part(datadir, '/', 2) || '/' || split_part(datadir, '/', 3) AS datadir FROM gp_segment_configuration WHERE role = 'p' AND content > 0;
with open("data_directories.txt", "w") as f:
    for row in result:
        f.write(row.datadir + "\n")

print(open("data_directories.txt").read())

## Create Table

This table will be used to verify that the backup was taken and restored.

In [ ]:
%%sql
DROP SCHEMA IF EXISTS foo CASCADE;
CREATE SCHEMA foo;

CREATE TABLE foo.bar AS SELECT i FROM generate_series(1, 100000) as i DISTRIBUTED BY (i);

SELECT *
FROM foo.bar
LIMIT 10;

In [ ]:
# Close the database connection
connection_url = f"postgresql://{PGUSER}@{PGHOST}:{PGPORT}/{PGDATABASE}"
%sql --close {{connection_url}}

## Disaster Recovery Utility
The `whpg-dr` commands are typically executed from the command line on the coordinator node but are being executed here in a Notebook for the demo.

### Create yaml configuration file
This file is used to configure backups to S3.

In [ ]:
config = f"""
cluster_name: {cluster_name}
storage:
  type: s3
  bucket: {bucket_name}
  prefix: {prefix}
  region: {region}

barman_options:
  compression: gzip
  compression_level: 6
"""

with open("whpg-dr_demo.yaml", "w") as f:
    f.write(config)

print(open("whpg-dr_demo.yaml").read())
    

## Configure Backup
Store the metadata needed for disaster recovery backups and enable WAL archiving.

In [ ]:
!echo y | whpg-dr configure backup /home/gpadmin/whpg-dr_demo.yaml

### Check 
Make sure the configure backup command worked properly.

In [ ]:
!whpg-dr check {cluster_name}

## Take a backup

In [ ]:
!whpg-dr backup {cluster_name}

## List backups

In [ ]:
!whpg-dr list-backup {cluster_name}

## Restore Point
At this point, an initial backup has been created and WAL archive is copied to S3. You can recover this cluster now.

A restore point provides point-time-recovery of the cluster. Changes to your cluster are tracked in WAL files and using the `create-restore-point` command provides a way to take a backup of just the changes (incremental) since the last full backup was taken.

Note: You don't need to create a restore point immediately after taking a backup.

In [ ]:
!whpg-dr create-restore-point {cluster_name}

## Additional Restore Point
After executing `create-restore-point`, you see that there is an additional point in time to restore the backup to.

You should see two Restore Points. The first was created with the full backup and the second was just taken moments later.

In [ ]:
!whpg-dr list-backup {cluster_name}

## Restore
A restore is typically performed on a different cluster but for this demo, we will use the same one. 
### Steps
* Create the restore configuration file
* Shutdown WarehousePG
* Delete all database files
* Restore latest Restore Point
* Promote the cluster
* Validate

In [ ]:
import os

coordinator = "cdw"

with open("data_directories.txt") as d:
    data_directories = "\n".join(f"  - {line.strip()}" for line in d if line.strip())

with open("segment_nodes.txt") as f:
    segment_hosts = "\n".join(f"  - {line.strip()}" for line in f if line.strip())
    
if os.path.isdir("/s3data"):
    coordinator_data_dir = "/data/coordinator"
else:
    coordinator_data_dir = "/data1/coordinator"

config = f"""source_cluster_name: {cluster_name}

storage:
  type: s3
  bucket: {bucket_name}
  prefix: {prefix}
  region: {region}
  credential_source: default

coordinator_host: {coordinator}
coordinator_data_directory: {coordinator_data_dir}

segment_hosts:
{segment_hosts}

data_directory: 
{data_directories}

data_directory_prefix: gpseg
"""

with open("restore.yaml", "w") as f:
    f.write(config)

print(open("restore.yaml").read())

## Remove the existing WarehousePG cluster
This step is only needed when you are restoring to an existing cluster. 

In [ ]:
import glob
import shutil
import os

!gpssh -f all_nodes.txt -e "killall loki" || true
!gpssh -f all_nodes.txt -e "killall wem" || true

!pxf cluster stop || true 
!gpstop -M immediate -a || true
!gpssh -f all_nodes.txt -e "rm -rf /data1/coordinator/* /data[1-8]/primary/* /data[1-8]/mirror*"

# Checks for /s3data used for the S3Files + EFS template
if os.path.isdir("/s3data"):
    for pattern in ["/s3data/whpg/*", "/data/coordinator/*", "/data/primary/*"]:
        for path in glob.glob(pattern):
            if os.path.isdir(path) and not os.path.islink(path):
                shutil.rmtree(path)
            else:
                os.remove(path)

## Configure Restore

In [ ]:
!echo y | whpg-dr configure restore /home/gpadmin/restore.yaml

## Restore Latest

In [ ]:
!echo y | whpg-dr restore {cluster_name} --target-name latest

## Cluster Status
The cluster is now available for additional WAL archives to be applied. In a disaster recovery configuration, the second cluster located in a different region would be ready for more Recovery Points to be applied or be promoted to a active cluster.

## Promote the Cluster
Make the cluster available.

In [ ]:
!whpg-dr promote {cluster_name}
!gpstart -a
!pxf cluster start

!sudo systemctl restart loki
!sudo systemctl restart wem


## Reconnect to the Database

In [ ]:
# Connect to the database
from sqlalchemy import create_engine
PGUSER="gpadmin"
PGHOST="cdw"
PGPORT="5432"
PGDATABASE="dev"
conn = create_engine(f"postgresql://{PGUSER}@{PGHOST}:{PGPORT}/{PGDATABASE}")

# If running elsewhere without trust auth, set a password and use this instead:
# PGPASSWORD="your_password_here"
# conn = create_engine(f"postgresql://{PGUSER}:{PGPASSWORD}@{PGHOST}:{PGPORT}/{PGDATABASE}")

%reload_ext sql
%sql conn

## Verify Cluster was Restored

In [ ]:
%%sql
SELECT *
FROM foo.bar
LIMIT 10;

## Close the Connection

In [ ]:
connection_url = f"postgresql://{PGUSER}@{PGHOST}:{PGPORT}/{PGDATABASE}"
%sql --close {{connection_url}}